# Bayesian Linear Regression with ADVI in PyTorch

Source:<P>
  https://luiarthur.github.io/statorial/varinf/linregpy/ <P>
  https://github.com/luiarthur/statorial/blob/master/docs/assets/varinf/python/LinReg.ipynb <P>

Adapted:

Antonio Esteves @UMinho, Fev 2024

## Introduction

In this notebook, we will demonstrate how to fit a Bayesian linear model using ADVI.

ADVI is a way of implementing variational inference without having to manually compute derivatives, 
using an automatic differentiation library. It is typically faster, and easier to implement
than tradititional MCMC methods. While, MCMC is still considered the gold standard in Bayesian applications,
variational inference enables the exploration of many models quickly. PyTorch is a python library 
that supports the creation of complex computational graphs and automatic differentiation.

***

## Implementation

First, we import the necessary libraries. The `torch` library is adopted for automatic differentiation (AD), but any other AD library can be used.

In [ ]:
import torch
from   torch.distributions import Normal, Gamma

import numpy as np
import matplotlib.pyplot as plt
from   tqdm import trange

# Define float64 the default type, instead of float32

torch.set_default_dtype(torch.float64)

Below, we implement the class `ModelParam`. We will see that `ModelParam` holds tensor `vp` and `size`, which is the dimensions of the model parameters.
When we instantiate an object of this class, the variational parameters are created: the mean and the log of standard deviation for a normal distribution. These parameters are optimized while we maximize the ELBO. It is used the log of standard deviation because it is not constrained, while standard deviation cannot be negative. This way we can optimize the log in an unconstrained fashion. Recall that in ADVI, the variational distribution parameters are transformed to the $\mathbb{R}$ space. The advantages of doing so include (1) being able to re-parameterize the distribution and being able to sample from a parameter free standard normal distribution, in order to calculate the gradients of random draws from the variational distribution; and (2) modeling correlation between parameters, if desired.

In [ ]:
class ModelParam():
    '''
    Class to model a parameter with a Normal distribution N(m,log_s).
    '''
    def __init__(self, size, m=None, log_s=None):
        if m is None:
            # if not specified, select a random mean for the unconstrained normal 
            # variational distribution.
            m = torch.randn(size)

        if log_s is None:
            # if not specified, select a random value for the log of standard deviation 
            # of the unconstrained normal variational distribution.
            log_s = torch.randn(size)

        # Set the variational parameters tensor
        self.vp = torch.stack([m, log_s])
        self.vp.requires_grad = True       # gradients of 'vp' will be calculated

        # Set the dimension of the variational parameters
        self.size = size

    def dist(self):
        '''
        Returns an unconstrained normal distribution parameterized by 
        the class parameters (m, e^log_s).

        NOTE: If a given parameter theta_i in theta has only a positive support,
               we convert it to log(theta_i) and specify the normal variational 
               distribution for log(theta_i).
        '''
        
        return torch.distributions.Normal(self.vp[0], self.vp[1].exp()) 

    def rsample(self, n=torch.Size([])):
        '''
        Returns 'n' samples for the parameter from the unconstrained normal distribution.
        Sampling uses the reprameterizing trick to allow gradient backpropagation.
        
        The returned sampled values are equivalent to those generated by:
           self.vp[0] + torch.randn(n) * self.vp[1].exp()
        where the sample is drawn from parameter-free standard normal, and
        then multiplied by standard deviation and added to the mean.
        '''
        return self.dist().rsample(n)

    def log_q(self, real):
        '''
        Returns the logarithm of the unconstrained variational density, evaluated 
        at position 'real'.
        '''
        return self.dist().log_prob(real).sum()


Our linear model is the following:

$$
\begin{aligned}
y_i \mid \sigma, \beta &\sim \text{Normal}(\beta_0 + \beta_1 x_i, \sigma), \text{ for } i = 1,\cdots, N\\
\beta_k &\sim \text{Normal}(0, 1), \text{ for } k = 0, 1 \\
\sigma &\sim \text{Gamma}(1, 1). \\
\end{aligned}
$$

We encode this model in the following cells. Note that we need to implement:

1. the log likelihood of the data: $\log p(data\mid T^{-1}(\tilde{\zeta}))$.
2. the log of the prior density $p(theta)$ times absolute value of the determinant of the jacobian (since we are transforming the parameters onto the real scale) $\vert \det J_{T^{-1}}(\tilde{\zeta})\vert$.
3. the log of the variational density, evaluated at the parameters sampled from the variational distribution: $\log q(\tilde{\zeta}; \theta)$.
4. the ELBO, which is calculated as follows:
    - sample model parameters (on the real scale) from the variational distributions;
        - recall that to obtain "parameterized" samples, we first draw from a standard normal, 
          and then we multiply by the standard deviation parameter of the variational normal distribution, 
          and then we add the mean parameter of the variational normal distribution.
    - transform the unconstrained parameters in $\mathbb{R}$ to their true support;
      in this case only $\sigma$ needs to be exponentiated.
    - Evaluate: ELBO = (1) + (2) - (3)

One more thing to note here is that we multiply the likelihood by the size of the full data, and divide by the size of the
current data. Hence the `mean(0)` in the return line of function `log_likelihood`. This enables stochastic variational inference to be done in minibatches. This can lead to huge speed-ups when analyzing large datasets.

In [ ]:
def log_likelihood(y, x, params, full_data_size):
    '''
    Returns the log likelihood of the data 'y': y ~ Normal(beta0+beta1*x, sigma).
    '''
    beta   = params['beta']
    sigma  = params['sigma']
    return Normal(x.matmul(beta), sigma).log_prob(y).mean(0) * full_data_size

def log_prior_plus_log_absdet_J(params_in_R, params):
    '''
    Returns: log p(beta1,beta2) + log |det J|
    '''
    # log of the prior for beta, evaluated at sampled values for beta
    lp_b = Normal(0, 1).log_prob(params_in_R['beta']).sum()

    # log of the prior for sigma + log of the Jacobian determinant: 
    #   log p(sigma) + det d(sigma) ---> log (det d(sigma)) = sigma 
    lp_log_sigma = (Gamma(1, 1).log_prob(params['sigma']) + params_in_R['sigma']).sum()

    return lp_b + lp_log_sigma


def log_q(model_params, params_in_R):
    '''
    Returns: log q(zeta~;phi)
    '''
    out = 0.0
    for key in model_params:
        out += model_params[key].log_q(params_in_R[key])
    return out

def elbo(y, x, model_params, full_data_size):
    
    params_in_R = {}
    
    # sample parameter values from their distributions
    for key in model_params:
        params_in_R[key] = model_params[key].rsample()

    # convert parameters in R to their true support
    params = {
        'beta':  params_in_R['beta'],
        'sigma': params_in_R['sigma'].exp()
    }

    # ELBO = log p(y,x) + log p(sigma) + det d(sigma) + log q(beta1,beta,sigma)
    out  = log_likelihood(y, x, params, full_data_size)
    out += log_prior_plus_log_absdet_J(params_in_R, params) 
    out -= log_q(model_params, params_in_R)

    return out

Now, we are ready to fit the model. For reproducibility, I will set seeds for the random number generators.

In [ ]:
# set random number generator seeds for reproducibility

torch.manual_seed(1)
np.random.seed(0)

We now generate some data. We will use 1000 samples. Note that the true values of the parameters are 
$\beta = (2, -3)$ and $\sigma = 0.5$. We can observe the plot of the data.

In [ ]:
# Generate data

N     = 1000
x     = torch.stack([torch.ones(N), torch.randn(N)], -1)
k     = x.shape[1]
beta  = torch.tensor([2., -3.])
sigma = 0.5
y     = Normal(x.matmul(beta), sigma).rsample()

# Plot the data

plt.scatter(x[:, 1].numpy(), y.numpy())
plt.xlabel("x")
plt.ylabel("y")
plt.show()

In next block of code we will optimize the model We first create a dictionary `model_params` with the model parameters.
All we have to do is specify the size for the initialization; the initial values for the variational parameters are drawn from a standard normal. Recall that the variational parameters are the mean and standard deviation of the Normal, but the standard deviation is stored as the logarithm of standard deviation in the dictionary.

We use the Adam optimizer with a learning rate of 0.1. This may seem large, but we will see that it is not.

We run stochastic gradient descent (SGD) for 1000 iterations. Each iteration, we subsample 100 observations from the original data. We use a loss equal to the negative of the ELBO divided by the size of the full dataset ($N$). Dividing the ELBO by $N$ allows us to use a larger learning rate.

In each iteration, we have to manually set the gradients to zero before computing the gradients with `loss.backward()`, since during optimization PyTorch accumulates the gradients.

In [ ]:
model_params = {
    'beta':  ModelParam(size=k),  # k=2 for params (beta1, beta2)
    'sigma': ModelParam(size=1)
} 
optimizer    = torch.optim.Adam([model_params[key].vp for key in model_params], lr=.1)
elbo_hist    = []

max_iter       = 3000
minibatch_size = 100

torch.manual_seed(1)

# Create a progress bar for SGD.
# max_iter - determines the final iteration.
# mininterval - determines how often the progress bar is updated (every 1 second here).

iters = trange(max_iter, mininterval=1)

# Stochastic gradient descent loop

for t in iters:
    sample_with_replacement = minibatch_size > N
    
    # get 'minibatch_size' IDs of dataset samples to use in the current iteration
    idx  = np.random.choice(N, minibatch_size, replace=sample_with_replacement)

    # Maximizing the ELBO is equivalent to minimizing -ELBO.
    # loss = -ELBO/N
    loss = -elbo(y[idx], x[idx, :], model_params, full_data_size=N) / N
    elbo_hist.append(-loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress bar
    iters.set_description('ELBO: {}'.format(elbo_hist[-1]), refresh=False)

Plot the elbo history, which seems to indicate convergence.

In [ ]:
# Plot ELBO history
plt.plot(elbo_hist)
plt.title('complete ELBO evolution')
plt.show()

# Plot ELBO history (after 100-th iteration)
plt.plot(elbo_hist[1000:])
plt.title('ELBO after 1000 iterations of optimization')
plt.show()

Next, we inspect the statistics of the posterior: posterior mean and standard deviation.<P>
The estimated values are consistent with the true values.

In [ ]:
# Inspect the posterior

nsamples = 1000
sigma_post = model_params['sigma'].rsample([nsamples]).exp().detach().numpy()

print(f'True beta1,beta:                {beta.detach().numpy()}')
print(f'beta1,beta2 mean:               {model_params["beta"].vp[0].detach().numpy()}')
print(f'beta1,beta2 standard deviation: {model_params["beta"].vp[1].exp().detach().numpy()}')
print(f'\nTrue sigma:                     {sigma}')
print(f'sigma mean:                     {sigma_post.mean()}')
print(f'sigma standard deviation:       {sigma_post.std()}')